# 🖥️ Computer Vision Lab 7 — Sequence Models: RNN, LSTM & GRU

**BITS F459 — Computer Vision**

Runnable companion to [`labsheet7.md`](https://github.com/Aritra/Computer_Vision/blob/main/labsheets/labsheet7.md). The labsheet carries the full
explanations and the theory; this notebook carries the code, in order, ready to run.

Sections 1–3 use `%%writefile` to create `seq_dataset.py`, `seq_model.py` and
`seq_train.py` as **real files on disk**, which later cells import as modules — the same
project layout as Lab 6, and the same files run from a terminal.

> ⚠️ **Educational use only. Nothing here is financial advice or a trading system.**
> Part of the point is to see how hard it is to extract a usable signal from market data,
> and how easy it is to fool yourself into thinking you have.

> ⚠️ **Check your kernel first.** This needs **Python (cv-dl)** plus `yfinance` and
> `pandas`. If cell 1 raises `ModuleNotFoundError`, you are on the wrong kernel — see
> Lab 6-1 Section 0.

---
## 0️⃣ Setup

```bash
conda activate cv-dl
pip install yfinance pandas
```

`sys.executable` must contain `cv-dl` — that is what proves the kernel is right.

In [ ]:
import sys, torch, pandas
print("python:", sys.executable)          # must contain 'cv-dl'
print("torch :", torch.__version__, "| pandas:", pandas.__version__)
import yfinance
print("yfinance imported OK")

Start TensorBoard **before** training anything, then refresh it as runs appear.
Useful tabs here: **SCALARS** (`loss/train`, `loss/val`, `metric/val`, `grad_norm/mean`,
`generalization_gap`) and **HPARAMS** (every run as one sortable row).

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

---
## 1️⃣ Stock Data: Returns, Windows and Leakage

### 1.1 Write `seq_dataset.py`

This module does the chronological split, computes targets **inside each segment**, fits
the scaler on **train only**, and builds windows — in that order, which is what keeps the
four leakage traps closed.

It also ships a **synthetic GARCH fallback**, so if the download fails you can still
complete every section. (The exercise asks you to say which data you used.)

In [ ]:
%%writefile seq_dataset.py
"""seq_dataset.py - stock-market windows, a synthetic fallback, and a memory task."""

import os

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset


# ------------------------------------------------------------------ generic
class TensorSeqDataset(Dataset):
    """Wraps pre-built tensors. features: (N, T, F); targets: (N,) or (N, 1)."""

    def __init__(self, features, targets):
        self.features = features
        self.targets = targets
        self.input_size = features.shape[2]
        self.seq_len = features.shape[1]

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]


# ------------------------------------------------------------- data loading
def make_synthetic_stock(n=2500, seed=0, start="2015-01-01"):
    """GARCH(1,1) returns: UNPREDICTABLE direction, PREDICTABLE volatility.

    Fallback when the download fails, and a ground-truth lab bench: by
    construction there is no return signal to find, but volatility clusters.
    """
    rng = np.random.default_rng(seed)
    omega, alpha, beta = 2e-6, 0.08, 0.90        # long-run daily vol = 1%
    var = omega / (1 - alpha - beta)
    r = np.zeros(n)
    for t in range(n):
        r[t] = np.sqrt(var) * rng.standard_normal()
        var = omega + alpha * r[t] ** 2 + beta * var
    close = 100 * np.exp(np.cumsum(r))
    volume = 1e6 * (1 + 20 * np.abs(r)) * np.exp(0.2 * rng.standard_normal(n))
    idx = pd.bdate_range(start, periods=n)
    return pd.DataFrame({"Close": close, "Volume": volume}, index=idx)


def load_prices(ticker="AAPL", start="2015-01-01", end=None, cache_dir="./data"):
    """Daily adjusted Close + Volume. Downloads once, then reads the cached CSV."""
    os.makedirs(cache_dir, exist_ok=True)
    safe = ticker.replace("^", "").replace(".", "_")
    path = os.path.join(cache_dir, f"{safe}_{start}.csv")
    if os.path.exists(path):
        return pd.read_csv(path, index_col=0, parse_dates=True)
    try:
        import yfinance as yf
        raw = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
        if raw.empty:
            raise RuntimeError("empty download (bad ticker, or no network?)")
        if isinstance(raw.columns, pd.MultiIndex):       # newer yfinance versions
            raw.columns = raw.columns.get_level_values(0)
        df = raw[["Close", "Volume"]].dropna()
        df.to_csv(path)                                  # cache REAL data only
        print(f"[load_prices] downloaded {ticker}: {len(df)} rows, "
              f"{df.index[0].date()} to {df.index[-1].date()}")
        return df
    except Exception as exc:
        print(f"[load_prices] WARNING: download failed for {ticker} "
              f"({type(exc).__name__}: {exc}).")
        print("[load_prices] Using SYNTHETIC data instead - say so in your report.")
        return make_synthetic_stock(start=start)        # not cached on purpose


# ----------------------------------------------------- targets and windowing
def _segment_targets(r, target, h_vol):
    """Targets for ONE segment, computed inside it (no peeking across splits).

    r[t] is the log-return of day t. A window ending at day t predicts:
      return    : r[t+1]
      direction : 1 if r[t+1] > 0 else 0
      volatility: log of the REALIZED volatility of r[t+1 .. t+h_vol],
                  i.e. the root-mean-square of those returns.
    The last rows without a full horizon get NaN and are dropped later.

    Why RMS and not std? Realized volatility is conventionally computed
    WITHOUT subtracting the mean (the daily mean return is ~0, and
    demeaning a handful of samples only adds noise). It also stays well
    defined at h_vol = 1, where it reduces to log|r[t+1]|; std() of a
    single sample is exactly 0, which would make every target the
    constant log(1e-8) = -18.42.
    """
    n = len(r)
    y = np.full(n, np.nan)
    if target in ("return", "direction"):
        nxt = r[1:]
        y[:-1] = nxt if target == "return" else (nxt > 0).astype(float)
    elif target == "volatility":
        for t in range(n - h_vol):
            seg = r[t + 1: t + 1 + h_vol]
            y[t] = np.log(np.sqrt((seg ** 2).mean()) + 1e-8)
    else:
        raise ValueError(f"unknown target {target!r}")
    return y


def _windows(feats, y, window):
    X, Y = [], []
    for end in range(window - 1, len(y)):
        if np.isnan(y[end]):
            continue
        X.append(feats[end - window + 1: end + 1])
        Y.append(y[end])
    return np.stack(X).astype(np.float32), np.asarray(Y)


def build_stock_datasets(prices, target="volatility", window=30, h_vol=5,
                         fracs=(0.7, 0.15, 0.15)):
    """Chronological split FIRST -> targets inside each segment -> scaler on TRAIN
    only -> windows. Returns (train_ds, val_ds, test_ds, scaler).

    Features per day (F = 2): log-return and change in log-volume, each divided
    by its TRAIN-segment std (no mean subtraction) and clipped to +/-8 for the
    heavy tails typical of markets.
    """
    close = prices["Close"].astype(float)
    volume = prices["Volume"].astype(float).replace(0, np.nan)
    frame = pd.DataFrame({"ret": np.log(close).diff(), "dvol": np.log(volume).diff()})
    frame = frame.replace([np.inf, -np.inf], np.nan).dropna()

    n = len(frame)
    a, b = int(n * fracs[0]), int(n * (fracs[0] + fracs[1]))
    segs = [frame.iloc[:a], frame.iloc[a:b], frame.iloc[b:]]

    ret_std = float(segs[0]["ret"].std())                # fitted on TRAIN only
    dvol_std = float(segs[0]["dvol"].std())
    scale = np.array([ret_std, dvol_std])

    raw_targets = [_segment_targets(s["ret"].to_numpy(), target, h_vol) for s in segs]

    t_mean, t_std = 0.0, 1.0
    if target == "return":
        t_std = ret_std                                  # keeps zero return at zero
    elif target == "volatility":
        tr = raw_targets[0][~np.isnan(raw_targets[0])]
        t_mean, t_std = float(tr.mean()), float(tr.std())

    datasets = []
    for seg, y in zip(segs, raw_targets):
        feats = np.clip(seg[["ret", "dvol"]].to_numpy() / scale, -8, 8)
        if target != "direction":
            y = (y - t_mean) / t_std
        X, Y = _windows(feats, y, window)
        if target == "direction":
            Y = torch.tensor(Y, dtype=torch.long)
        else:
            Y = torch.tensor(Y, dtype=torch.float32)[:, None]
        datasets.append(TensorSeqDataset(torch.tensor(X), Y))

    scaler = dict(target=target, window=window, h_vol=h_vol, ret_std=ret_std,
                  dvol_std=dvol_std, t_mean=t_mean, t_std=t_std)
    return (*datasets, scaler)


# ----------------------------------------------------------------- baselines
def baseline_scores(test_ds, scaler, train_ds=None):
    """The bar a model must clear. Regression numbers are MSE on the SCALED target.

    return    : zero_mse (always predict no change), prev_return_mse
    direction : majority_acc (always predict the train majority class), prev_sign_acc
    volatility: train_mean_mse (predict the average), recent_vol_mse (persistence:
                'the next 5 days will be as volatile as the last 5')
    """
    target, X, y = scaler["target"], test_ds.features, test_ds.targets
    out = {}
    if target == "return":
        out["zero_mse"] = (y ** 2).mean().item()
        out["prev_return_mse"] = ((X[:, -1, 0:1] - y) ** 2).mean().item()
    elif target == "direction":
        majority = int(train_ds.targets.float().mean() >= 0.5)
        out["majority_acc"] = (y == majority).float().mean().item()
        out["prev_sign_acc"] = ((X[:, -1, 0] > 0).long() == y).float().mean().item()
    else:
        h = scaler["h_vol"]
        recent = X[:, -h:, 0] * scaler["ret_std"]        # raw returns, last h days
        # RMS, to match the target definition in _segment_targets
        pred = (torch.log(recent.pow(2).mean(dim=1).sqrt() + 1e-8)
                - scaler["t_mean"]) / scaler["t_std"]
        out["train_mean_mse"] = (y ** 2).mean().item()   # predicting 0 on the scaled axis
        out["recent_vol_mse"] = ((pred[:, None] - y) ** 2).mean().item()
    return out


def skill(model_mse, baseline_mse):
    """Out-of-sample skill = 1 - MSE_model / MSE_baseline. 0 = no better, <0 = worse."""
    return 1.0 - model_mse / baseline_mse


# ------------------------------------------------------ walk-forward folds
def walk_forward(prices, n_folds=4, val_len=250, test_len=250):
    """Yield (fold, prices_up_to_fold_end, fracs) for expanding-window evaluation.

    Each fold trains on everything before its validation block, validates on the
    next `val_len` days and tests on the `test_len` days after that. Pass the
    yielded prices and fracs straight to build_stock_datasets(). (Boundaries are
    approximate because 1 row is lost to differencing.)
    """
    n = len(prices)
    for k in range(n_folds):
        end = n - (n_folds - 1 - k) * test_len
        fracs = ((end - val_len - test_len) / end, val_len / end, test_len / end)
        yield k, prices.iloc[:end], fracs


# ------------------------------------------------- memory task (Section 5.2)
def make_first_token_task(n, T, k=2, seed=0):
    """The label is hidden in the FIRST time step; the other T-1 steps are noise.

    Channel 0 is pure noise at every step. Channel 1 is zero except at t=0, where
    it encodes the class. To classify correctly the network must carry that one
    signal across T-1 steps of distraction. Chance accuracy = 1/k.
    """
    g = torch.Generator().manual_seed(seed)
    labels = torch.randint(0, k, (n,), generator=g)
    x = torch.randn(n, T, 2, generator=g)
    x[:, :, 1] = 0.0
    x[:, 0, 1] = labels.float() - (k - 1) / 2
    return TensorSeqDataset(x, labels)

### 1.2 The persistence illusion

Run this once, because it is the most common beginner mistake in financial ML. You will
get an R² very close to 1 — which does **not** mean prices are predictable.

In [ ]:
import numpy as np
from seq_dataset import load_prices

df = load_prices("AAPL", start="2015-01-01")
close = df["Close"].to_numpy()

# "Forecast" tomorrow's PRICE as today's price. No learning at all.
mse = np.mean((close[1:] - close[:-1]) ** 2)
r2 = 1 - mse / np.var(close[1:])
print(f"R^2 of 'tomorrow = today' on PRICE LEVELS: {r2:.4f}")

### 1.3 Task 1 — prices vs returns

Plot `Close` and the log-return series side by side. Which looks stationary?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from seq_dataset import load_prices

df = load_prices("AAPL", start="2015-01-01")
close = df["Close"].to_numpy()
ret = np.diff(np.log(close))

fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
ax[0].plot(df.index, close, lw=0.8)
ax[0].set_title("Close (level) - trends, drifts, non-stationary")
ax[1].plot(df.index[1:], ret, lw=0.4)
ax[1].set_title("log-return - roughly stationary, volatility clusters")
for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 1.4 Task 3 — volatility clustering

Autocorrelation of `r_t` versus `|r_t|` at lags 1–20. **This single plot is the empirical
justification for the whole lab:** whichever series shows structure is the one that is
forecastable.

In [ ]:
def acf(x, k):
    return np.corrcoef(x[:-k], x[k:])[0, 1]

lags = range(1, 21)
acf_r = [acf(ret, k) for k in lags]
acf_abs = [acf(np.abs(ret), k) for k in lags]

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar([l - 0.2 for l in lags], acf_r, width=0.4, label="r_t  (returns)")
ax.bar([l + 0.2 for l in lags], acf_abs, width=0.4, label="|r_t|  (magnitude)")
ax.axhline(0, color="k", lw=0.8)
# rough 95% band for white noise
band = 1.96 / np.sqrt(len(ret))
ax.axhline(band, ls="--", c="grey", lw=0.8)
ax.axhline(-band, ls="--", c="grey", lw=0.8)
ax.set_xlabel("lag (trading days)")
ax.set_ylabel("autocorrelation")
ax.set_title("Returns look like noise; their MAGNITUDE does not")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"mean |acf| of r_t   over lags 1-20: {np.mean(np.abs(acf_r)):.4f}")
print(f"mean |acf| of |r_t| over lags 1-20: {np.mean(np.abs(acf_abs)):.4f}")
print(f"95% white-noise band: +/- {band:.4f}")

### 1.5 Task 4 + 6 — build the three datasets, and the baselines

**Predict before you run:** will "recent volatility" beat "predict the average" for the
volatility target, and by roughly how much? Write your guess down, *then* look.

In [ ]:
from seq_dataset import build_stock_datasets, baseline_scores

for target in ("return", "direction", "volatility"):
    tr, va, te, scaler = build_stock_datasets(df, target=target, window=30)
    base = baseline_scores(te, scaler, tr)
    print(f"{target:11s} train X{tuple(tr.features.shape)} y{tuple(tr.targets.shape)}"
          f"  | val {len(va):4d} | test {len(te):4d}")
    print(f"            baselines: {({k: round(v, 4) for k, v in base.items()})}")

> 🔎 **Read the volatility baselines carefully.** Compare `recent_vol_mse` against
> `train_mean_mse`. If persistence is *worse* than simply predicting the average, then a
> large "skill vs recent-vol" number does **not** mean your model is good — it means the
> opponent is weak. Always report skill against **both** baselines, as the exercise requires.

---
## 2️⃣ The Models: RNN, LSTM, GRU

> ⚠️ **The classic bidirectional bug.** With `bidirectional=True`, `out[:, -1]` is *not*
> the final state of the backward pass — that pass ends at the first element, so at index
> `-1` it has seen a single input. Use `h_n`. `SeqModel` below does this correctly.

In [ ]:
%%writefile seq_model.py
"""seq_model.py - RNN / LSTM / GRU wrapped in one configurable class."""

import torch
from torch import nn


class SeqModel(nn.Module):
    """(B, T, F) -> (B, out_features).

    cell          : "rnn" | "lstm" | "gru"
    pooling       : "last" (final hidden state h_n) | "mean" (mean of out over time)
    out_features  : 1 for regression, num_classes for classification
    Returns raw outputs (logits for classification). Softmax belongs at display
    time, as in Lab 6-1 - never inside forward().
    """

    def __init__(self, cell="lstm", input_size=1, hidden_size=64, num_layers=1,
                 bidirectional=False, dropout=0.0, out_features=1,
                 pooling="last", nonlinearity="tanh"):
        super().__init__()
        classes = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}
        kwargs = dict(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True, bidirectional=bidirectional,
            # nn.* applies this BETWEEN layers only; with 1 layer PyTorch would
            # warn and ignore it, so we zero it ourselves.
            dropout=dropout if num_layers > 1 else 0.0,
        )
        if cell == "rnn":
            kwargs["nonlinearity"] = nonlinearity
        self.cell = cell
        self.bidirectional = bidirectional
        self.pooling = pooling
        self.rnn = classes[cell](**kwargs)
        self.drop = nn.Dropout(dropout)          # dropout before the output layer
        width = hidden_size * (2 if bidirectional else 1)
        self.head = nn.Linear(width, out_features)

    def forward(self, x):
        out, hn = self.rnn(x)
        if isinstance(hn, tuple):                # LSTM returns (h_n, c_n)
            hn = hn[0]
        if self.pooling == "last":
            summary = torch.cat([hn[-2], hn[-1]], dim=1) if self.bidirectional else hn[-1]
        elif self.pooling == "mean":
            summary = out.mean(dim=1)
        else:
            raise ValueError(f"unknown pooling {self.pooling!r}")
        return self.head(self.drop(summary))


def init_forget_bias(model, value=1.0):
    """Set the LSTM forget-gate bias to `value` (a common trick: start by remembering).

    PyTorch packs LSTM gates in the order [input, forget, cell, output], so the
    forget gate is the second quarter of each bias vector.
    """
    H = model.rnn.hidden_size
    with torch.no_grad():
        for name, p in model.rnn.named_parameters():
            if name.startswith("bias_ih"):
                p[H:2 * H].fill_(value)
            elif name.startswith("bias_hh"):
                p[H:2 * H].zero_()     # keep the total forget bias equal to `value`


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ---------------------------------------------------------------- from scratch
class ManualLSTM(nn.Module):
    """A single-layer, unidirectional LSTM written out gate by gate.

    Parameter layout deliberately mirrors nn.LSTM (gate order i, f, g, o), so
    you can copy weights across and check the outputs agree.
    """

    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.H = hidden_size
        k = 1.0 / hidden_size ** 0.5
        self.weight_ih = nn.Parameter(torch.empty(4 * hidden_size, input_size).uniform_(-k, k))
        self.weight_hh = nn.Parameter(torch.empty(4 * hidden_size, hidden_size).uniform_(-k, k))
        self.bias_ih = nn.Parameter(torch.empty(4 * hidden_size).uniform_(-k, k))
        self.bias_hh = nn.Parameter(torch.empty(4 * hidden_size).uniform_(-k, k))

    def forward(self, x):                        # x: (B, T, F)
        B, T, _ = x.shape
        h = x.new_zeros(B, self.H)
        c = x.new_zeros(B, self.H)
        outputs = []
        for t in range(T):
            gates = (x[:, t] @ self.weight_ih.T + self.bias_ih
                     + h @ self.weight_hh.T + self.bias_hh)
            i, f, g, o = gates.chunk(4, dim=1)
            i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
            g = torch.tanh(g)
            c = f * c + i * g                    # additive cell-state update
            h = o * torch.tanh(c)
            outputs.append(h)
        return torch.stack(outputs, dim=1), (h, c)

### Task 1 — parameter counting

For one unidirectional layer the count is `g·(H·I + H·H + 2H)`, with `g` = 1, 3, 4 for
RNN, GRU, LSTM. Verify, then work out what bidirectional and a second layer do.

In [ ]:
import torch
from torch import nn
from seq_model import SeqModel, ManualLSTM, count_params, init_forget_bias

I, H = 2, 32
print(f"{'cell':6s} {'g':>2s} {'formula':>8s} {'actual':>8s}  match")
for cell, g in [("rnn", 1), ("gru", 3), ("lstm", 4)]:
    m = SeqModel(cell=cell, input_size=I, hidden_size=H, out_features=1)
    actual = sum(p.numel() for p in m.rnn.parameters())
    formula = g * (H * I + H * H + 2 * H)
    print(f"{cell:6s} {g:2d} {formula:8d} {actual:8d}  {formula == actual}")

# TODO: repeat for bidirectional=True and num_layers=2. What is the rule?

### Task 2 — verify the gates: `ManualLSTM` vs `nn.LSTM`

Copy the four parameter tensors across and confirm the outputs agree. If this passes, your
hand-written gate equations are exactly what PyTorch computes.

In [ ]:
torch.manual_seed(0)
ref = nn.LSTM(2, 16, batch_first=True)
man = ManualLSTM(2, 16)

man.weight_ih.data.copy_(ref.weight_ih_l0.data)
man.weight_hh.data.copy_(ref.weight_hh_l0.data)
man.bias_ih.data.copy_(ref.bias_ih_l0.data)
man.bias_hh.data.copy_(ref.bias_hh_l0.data)

x = torch.randn(4, 7, 2)
out_ref, (h_ref, c_ref) = ref(x)
out_man, (h_man, c_man) = man(x)

print("outputs match :", torch.allclose(out_ref, out_man, atol=1e-6))
print("h_n match     :", torch.allclose(h_ref[0], h_man, atol=1e-6))
print("c_n match     :", torch.allclose(c_ref[0], c_man, atol=1e-6))

### Tasks 3 and 4 — forget gate, and why repeated multiplication hurts

In [ ]:
print("Task 4, with numbers:")
print(f"  0.9**100 = {0.9**100:.3e}   <- vanishing")
print(f"  1.1**100 = {1.1**100:.3e}   <- exploding")
print(f"  0.5**50  = {0.5**50:.3e}    <- an LSTM forget gate left at sigmoid(0)=0.5")

# Task 3: force the forget gate open / shut inside ManualLSTM and describe the cell state.
# Hint: f is the SECOND quarter of the gate vector (PyTorch order is i, f, g, o).
# TODO: your experiment here.

---
## 3️⃣ `seq_train.py`: One Parameterized Training Function

Same idea as Lab 6: one function configured by a dict, TensorBoard logging, early
stopping, best-checkpointing. New here: **gradient clipping** and a **gradient-norm log**.

Note `max_norm = float("inf")` when `clip` is unset — that makes `clip_grad_norm_` a pure
*measurement*, so `grad_norm/mean` is logged for every run whether or not you clip.

In [ ]:
%%writefile seq_train.py
"""seq_train.py"""

import os
import time

import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

from seq_model import SeqModel, init_forget_bias, count_params


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


@torch.no_grad()
def evaluate(model, loader, criterion, task, device):
    """Returns (loss, metric): metric is accuracy for classification, MAE for regression.

    Calls model.eval() itself - exactly like Lab 6-1's evaluate().
    """
    model.eval()
    total_loss = metric = n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        bs = len(y)
        total_loss += criterion(out, y).item() * bs
        if task == "classification":
            metric += (out.argmax(dim=1) == y).sum().item()
        else:
            metric += (out - y).abs().mean(dim=1).sum().item()
        n += bs
    return total_loss / n, metric / n


def build_optimizer(name, params, lr, weight_decay):
    if name == "adam":
        return torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)
    if name == "adamw":
        return torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    if name == "sgd":
        return torch.optim.SGD(params, lr=lr, weight_decay=weight_decay)
    if name == "sgd_momentum":
        return torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay)
    raise ValueError(f"unknown optimizer {name!r}")


def train(config, train_ds, val_ds, test_ds=None, extras=None):
    """`extras` (e.g. the scaler dict) is stored in best.pt for inference."""
    torch.manual_seed(config.get("seed", 0))
    device = get_device()
    task = config["task"]

    model = SeqModel(
        cell=config["cell"], input_size=config["input_size"],
        hidden_size=config.get("hidden_size", 64),
        num_layers=config.get("num_layers", 1),
        bidirectional=config.get("bidirectional", False),
        dropout=config.get("dropout", 0.0),
        out_features=config["out_features"],
        pooling=config.get("pooling", "last"),
    ).to(device)
    if config.get("forget_bias") is not None and config["cell"] == "lstm":
        init_forget_bias(model, config["forget_bias"])

    criterion = nn.MSELoss() if task == "regression" else nn.CrossEntropyLoss()
    optimizer = build_optimizer(config.get("optimizer", "adam"), model.parameters(),
                                config.get("lr", 1e-3), config.get("weight_decay", 0.0))

    bs = config.get("batch_size", 64)
    # Shuffling is safe HERE: the split was already chronological, and each
    # window is an independent (input, target) pair.
    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=512)
    test_loader = DataLoader(test_ds, batch_size=512) if test_ds is not None else None

    run_name = config["run_name"]
    writer = SummaryWriter(os.path.join("runs", run_name))
    ckpt_dir = os.path.join("checkpoints", run_name)
    os.makedirs(ckpt_dir, exist_ok=True)
    ckpt_path = os.path.join(ckpt_dir, "best.pt")

    # float("inf") makes clip_grad_norm_ a pure measurement (no clipping)
    max_norm = config["clip"] if config.get("clip") else float("inf")
    patience = config.get("patience", 5)

    best_val, best_epoch, bad_epochs = float("inf"), 0, 0
    history = []
    t_start = time.perf_counter()

    for epoch in range(1, config.get("epochs", 30) + 1):
        model.train()
        run_loss = run_gn = n_batches = n_seen = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            gn = nn.utils.clip_grad_norm_(model.parameters(), max_norm)
            optimizer.step()
            run_loss += loss.item() * len(y)
            n_seen += len(y)
            run_gn += float(gn)
            n_batches += 1

        train_loss = run_loss / n_seen
        val_loss, val_metric = evaluate(model, val_loader, criterion, task, device)
        mean_gn = run_gn / n_batches

        writer.add_scalar("loss/train", train_loss, epoch)
        writer.add_scalar("loss/val", val_loss, epoch)
        writer.add_scalar("metric/val", val_metric, epoch)
        writer.add_scalar("grad_norm/mean", mean_gn, epoch)
        writer.add_scalar("generalization_gap", val_loss - train_loss, epoch)
        history.append(dict(epoch=epoch, train_loss=train_loss, val_loss=val_loss,
                            val_metric=val_metric, grad_norm=mean_gn))

        if val_loss < best_val:
            best_val, best_epoch, bad_epochs = val_loss, epoch, 0
            torch.save({"model_state_dict": model.state_dict(), "config": config,
                        "epoch": epoch, "val_loss": val_loss, "val_metric": val_metric,
                        "extras": extras}, ckpt_path)
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    train_time = time.perf_counter() - t_start
    stopped_epoch = epoch

    # Report TEST performance of the BEST checkpoint, not of the last epoch.
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    result = dict(run_name=run_name, best_epoch=best_epoch, stopped_epoch=stopped_epoch,
                  best_val_loss=best_val, val_metric=ckpt["val_metric"],
                  train_time=train_time, n_params=count_params(model), history=history)
    if test_loader is not None:
        result["test_loss"], result["test_metric"] = evaluate(
            model, test_loader, criterion, task, device)

    writer.add_hparams({k: str(v) for k, v in config.items()},
                       {"hparam/val_loss": best_val,
                        "hparam/test_metric": result.get("test_metric", float("nan"))})
    writer.close()
    return result

> 💡 **`task` vs `target`.** In `train()`'s config, `task` is `"regression"` or
> `"classification"` (how the loss is computed). In `build_stock_datasets`, `target` is
> `"return"`, `"direction"` or `"volatility"` (what to predict). Return and volatility are
> regression; direction is classification.

> 💡 **Regression losses are on the scaled target**, so `test_loss` is directly comparable
> with `baseline_scores`. Never compare against a baseline computed on a different scale.

---
## 4️⃣ Worked Example: LSTM on Apple, Three Targets

The complete example. Expect roughly: **volatility** shows something real, **return**
skill ≈ 0 (possibly slightly negative), **direction** ≈ majority-class accuracy.

A model that honestly shows no skill on returns is a **correct result**, not a bug.

In [ ]:
"""run_stock_example.py - one LSTM, three targets, honest baselines."""

from seq_dataset import load_prices, build_stock_datasets, baseline_scores, skill
from seq_train import train

TICKER, START, WINDOW = "AAPL", "2015-01-01", 30

prices = load_prices(TICKER, start=START)
print(prices.tail(3))
print(f"{len(prices)} trading days, {prices.index[0].date()} to {prices.index[-1].date()}")

# stock target -> (how train() computes loss, number of outputs)
settings = {
    "volatility": dict(task="regression", out_features=1),
    "return":     dict(task="regression", out_features=1),
    "direction":  dict(task="classification", out_features=2),
}

for target, kind in settings.items():
    train_ds, val_ds, test_ds, scaler = build_stock_datasets(
        prices, target=target, window=WINDOW)
    base = baseline_scores(test_ds, scaler, train_ds)

    config = dict(
        run_name=f"stock_{TICKER}_{target}_lstm", cell="lstm", input_size=2,
        hidden_size=32, num_layers=1, dropout=0.0, optimizer="adam", lr=1e-3,
        weight_decay=1e-4, clip=1.0, batch_size=64, epochs=40, patience=8,
        window=WINDOW, **kind,
    )
    res = train(config, train_ds, val_ds, test_ds, extras=scaler)

    print(f"\n=== {target} ===  best epoch {res['best_epoch']} / stopped {res['stopped_epoch']}")
    print("baselines:", {k: round(v, 4) for k, v in base.items()})
    if target == "direction":
        print(f"LSTM test accuracy: {res['test_metric']:.4f}")
    else:
        ref = base["recent_vol_mse"] if target == "volatility" else base["zero_mse"]
        print(f"LSTM test MSE: {res['test_loss']:.4f}   skill vs "
              f"{'recent-vol' if target == 'volatility' else 'zero'} baseline: "
              f"{skill(res['test_loss'], ref):+.3f}")

### Try-it-yourself checklist

1. Change `TICKER` to `"^NSEI"` or `"RELIANCE.NS"`. Do the three conclusions survive?
2. Set `cell="gru"` and `cell="rnn"`. Does the ranking change for volatility? For return?
3. Change `WINDOW` to `10` and `60`. Does a longer window help volatility?
4. Set `TICKER = "NO_SUCH_TICKER"` to force the **synthetic** series. It has no return
   signal by construction and strong volatility clustering. Do your conclusions get
   *clearer*? Why does a ground-truth series help you judge your own pipeline?

---
## 5️⃣ Experiments

Every experiment is a config change with a new `run_name`, overlaid in TensorBoard.

### 5.1 RNN vs LSTM vs GRU

> ⚠️ **Compare against the baselines, not against each other.** Differences between cells
> are often smaller than the difference from changing only `seed`.

In [ ]:
import numpy as np
from seq_dataset import build_stock_datasets, baseline_scores, skill
from seq_train import train

TICKER, WINDOW = "AAPL", 30
settings = {
    "volatility": dict(task="regression", out_features=1),
    "return":     dict(task="regression", out_features=1),
    "direction":  dict(task="classification", out_features=2),
}

rows = []
for target, kind in settings.items():
    tr, va, te, scaler = build_stock_datasets(df, target=target, window=WINDOW)
    base = baseline_scores(te, scaler, tr)
    for cell in ("rnn", "lstm", "gru"):
        scores = []
        for seed in (0, 1, 2):
            cfg = dict(run_name=f"c51_{target}_{cell}_s{seed}", cell=cell, input_size=2,
                       hidden_size=32, optimizer="adam", lr=1e-3, weight_decay=1e-4,
                       clip=1.0, batch_size=64, epochs=40, patience=8, seed=seed, **kind)
            res = train(cfg, tr, va, te, extras=scaler)
            scores.append(res["test_metric"] if target == "direction" else res["test_loss"])
        rows.append((target, cell, np.mean(scores), np.std(scores)))
        print(f"{target:11s} {cell:5s} {np.mean(scores):.4f} +/- {np.std(scores):.4f}")
    print(f"            baselines: {({k: round(v, 4) for k, v in base.items()})}\n")

**Task.** Is any gap between two cells larger than the seed-to-seed spread? If not, say
so — "no detectable difference" is a valid and important result. Convert each regression
result to `skill()` against **both** volatility baselines.

### 5.2 Watching gradients vanish

Real markets are too noisy to isolate this, so Part A uses a controlled task: the label is
hidden at `t = 0` and the other `T − 1` steps are pure noise. Chance = 0.5.

> ⚠️ **`patience = epochs` here on purpose.** This task sits at chance for many epochs and
> then jumps suddenly. Any smaller patience early-stops during that plateau and you will
> wrongly record a failure.

In [ ]:
import numpy as np

from seq_dataset import make_first_token_task
from seq_train import train

# patience == epochs ON PURPOSE: this task sits at chance for many epochs and
# then jumps suddenly. Any smaller patience early-stops during that plateau and
# records a failure that never happened.
mem = dict(task="classification", input_size=2, out_features=2, hidden_size=32,
           optimizer="adam", lr=1e-3, batch_size=64, epochs=40, patience=40)

# The fourth variant is the point of the experiment, not an afterthought: an
# LSTM whose forget gate starts at sigmoid(0) ~ 0.5 decays its cell state by
# about half per step, so at T=50 roughly 0.5**50 ~ 1e-15 of the signal
# survives. forget_bias=1.0 starts the gate near "remember" instead.
variants = [("rnn", {}), ("lstm", {}), ("gru", {}), ("lstm_fb1", {"forget_bias": 1.0})]
SEEDS = (0, 1, 2)      # single-seed results on this task are a coin flip - see below

for T in (10, 25, 50, 100, 200):
    tr = make_first_token_task(2000, T, seed=1)
    va = make_first_token_task(500, T, seed=2)
    te = make_first_token_task(500, T, seed=3)
    for name, extra in variants:
        cell = "lstm" if name.startswith("lstm") else name
        accs = []
        for seed in SEEDS:
            res = train({**mem, "run_name": f"mem_{name}_T{T}_s{seed}",
                         "cell": cell, "seed": seed, **extra}, tr, va, te)
            accs.append(res["test_metric"])
        print(f"T={T:3d}  {name:8s}  {np.mean(accs):.3f} +/- {np.std(accs):.3f}  "
              f"(runs: {[round(a, 3) for a in accs]})")

Plot test accuracy against `T`, one line per cell.

In [ ]:
import matplotlib.pyplot as plt

# Fill this from the loop above (or collect into a dict while it runs).
# results[cell] = [acc at T=10, 25, 50, 100, 200]
Ts = [10, 25, 50, 100, 200]
results = {}        # TODO: populate

for cell, accs in results.items():
    plt.plot(Ts, accs, marker="o", label=cell)
plt.axhline(0.5, ls="--", c="grey", label="chance")
plt.xlabel("sequence length T")
plt.ylabel("test accuracy")
plt.title("How far back can each cell carry one bit?")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Part B: measure the gradient directly** — how much does the loss change if you nudge
step `t`'s input? That is `|∂loss/∂x_t|`.

In [ ]:
import torch
from torch import nn
from seq_model import SeqModel
from seq_dataset import make_first_token_task


def grad_per_timestep(model, ds, n=256):
    """Mean |d loss / d x_t| for each t, over n examples."""
    # NOTE: train mode on purpose. On a GPU, cuDNN's RNN backward raises an error
    # in eval mode. This is safe here because dropout=0 in this experiment.
    model.train()
    x = ds.features[:n].clone().requires_grad_(True)
    y = ds.targets[:n]
    loss = nn.functional.cross_entropy(model(x), y)
    loss.backward()
    return x.grad.abs().mean(dim=(0, 2))          # shape (T,)


T = 100
ds = make_first_token_task(512, T, seed=1)
profiles = {}
for cell in ("rnn", "lstm", "gru"):
    torch.manual_seed(0)
    m = SeqModel(cell=cell, input_size=2, hidden_size=32, out_features=2)
    profiles[cell] = grad_per_timestep(m, ds)     # at INITIALISATION, untrained

In [ ]:
# Plot the profiles with a LOG y-axis - the whole point is orders of magnitude.
plt.figure(figsize=(8, 4))
for cell, prof in profiles.items():
    plt.semilogy(prof.detach().numpy(), label=cell)
plt.xlabel("time step t   (the label lives at t = 0)")
plt.ylabel("mean |d loss / d x_t|   (log scale)")
plt.title(f"Gradient reaching each step, T={T}")
plt.legend()
plt.grid(alpha=0.3, which="both")
plt.show()

for cell, prof in profiles.items():
    print(f"{cell:5s} grad[0]={prof[0]:.3e}  grad[T-1]={prof[-1]:.3e}  "
          f"ratio={prof[-1] / prof[0]:.3e}")

> ⚠️ **These profiles are taken at *initialisation*, on an untrained model — read the
> result carefully before concluding anything.** An untrained LSTM has its forget gate at
> `sigmoid(0) ≈ 0.5`, so its cell state decays by about half per step (`0.5**50 ≈ 1e-15`).
> At initialisation an LSTM can therefore decay **as fast as, or faster than**, a vanilla
> RNN, and the ratio you measure may not favour the LSTM at all.
>
> The LSTM's long-range advantage comes from **learned** gates (a forget gate driven
> towards 1), not from its initial state. That is precisely why `forget_bias=1.0` exists.
> **Profile a trained model as well as an untrained one and compare the two** — the
> contrast is the real lesson here, and it is worth more than either number alone.

### 5.3 LSTM variants: depth, direction, pooling

Sweep one factor at a time from the base `cell="lstm", num_layers=1, bidirectional=False,
pooling="last"`, on the volatility target with three seeds.

In [ ]:
variants = [
    ("base",        {}),
    ("layers2",     {"num_layers": 2}),
    ("layers3",     {"num_layers": 3}),
    ("bidir",       {"bidirectional": True}),
    ("mean_pool",   {"pooling": "mean"}),
    ("forget_b1",   {"forget_bias": 1.0}),
]

tr, va, te, scaler = build_stock_datasets(df, target="volatility", window=30)
base = baseline_scores(te, scaler, tr)
print("baselines:", {k: round(v, 4) for k, v in base.items()})
print()
print(f"{'variant':12s} {'params':>8s} {'skill_recent':>13s} {'skill_mean':>11s}")

for name, extra in variants:
    losses, n_par = [], None
    for seed in (0, 1, 2):
        cfg = dict(run_name=f"c53_{name}_s{seed}", task="regression", cell="lstm",
                   input_size=2, out_features=1, hidden_size=32, optimizer="adam",
                   lr=1e-3, weight_decay=1e-4, clip=1.0, epochs=40, patience=8,
                   seed=seed, **extra)
        res = train(cfg, tr, va, te, extras=scaler)
        losses.append(res["test_loss"])
        n_par = res["n_params"]
    m = float(np.mean(losses))
    print(f"{name:12s} {n_par:8d} {skill(m, base['recent_vol_mse']):13.3f} "
          f"{skill(m, base['train_mean_mse']):11.3f}")

**Task 4 — reproduce the bidirectional bug.** Subclass `SeqModel` and use `out[:, -1]`
instead of `h_n` for the `"last"` pooling, then compare skill against the correct version.
This is the bug warned about in Section 2.

In [ ]:
from seq_model import SeqModel


class BuggyBiModel(SeqModel):
    """Deliberately WRONG: uses out[:, -1] on a bidirectional model.

    The backward pass ends at the FIRST element, so at index -1 it has seen
    exactly one input. Half the summary vector is therefore nearly content-free.
    """

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(self.drop(out[:, -1]))


# TODO: train BuggyBiModel vs the correct bidirectional SeqModel on the volatility
# target and compare skill. How much does the bug cost?

### 5.4 Gradient clipping

Recurrent gradients can also **explode**. Take the memory task at `T = 50` with `rnn` and
an aggressive setting (`optimizer="sgd"`, `lr=1e-2`), and sweep the clip threshold.

In [ ]:
from seq_dataset import make_first_token_task

T = 50
tr_m = make_first_token_task(2000, T, seed=1)
va_m = make_first_token_task(500, T, seed=2)
te_m = make_first_token_task(500, T, seed=3)

for clip in (None, 5.0, 1.0, 0.1):
    cfg = dict(run_name=f"c54_clip_{clip}", task="classification", cell="rnn",
               input_size=2, out_features=2, hidden_size=32, optimizer="sgd",
               lr=1e-2, batch_size=64, epochs=30, patience=30, clip=clip)
    res = train(cfg, tr_m, va_m, te_m)
    print(f"clip={str(clip):>5s}  test_acc={res['test_metric']:.3f}  "
          f"best_epoch={res['best_epoch']}")

# Then open grad_norm/mean in TensorBoard for all four runs and compare.

**Task 3.** `clip_grad_norm_` runs **after `backward()` and before `step()`**. Move it
after `step()` in a copy of the loop, show it has no effect, and explain why.

### 5.5 Overfitting, dropout, and `eval()` again

> ⚠️ Write your own evaluation helper for the `eval()` experiment. `seq_train.evaluate()`
> calls `model.eval()` **itself**, which would silently repair the very bug you are trying
> to show — exactly the trap from Lab 6-2.

In [ ]:
import torch
from torch.utils.data import DataLoader


@torch.no_grad()
def metric_leaving_mode_alone(model, ds, task="regression"):
    """Does NOT call model.eval() - that is the entire point."""
    loader = DataLoader(ds, batch_size=512)
    total = n = 0
    for x, y in loader:
        out = model(x)
        if task == "classification":
            total += (out.argmax(dim=1) == y).sum().item()
        else:
            total += (out - y).abs().mean(dim=1).sum().item()
        n += len(y)
    return total / n


# TODO: train a 2-layer LSTM with dropout=0.5 on a SMALL training set
# (e.g. the first 400 windows), then evaluate 5x in model.train() mode
# and 5x in model.eval() mode using the helper above. Report both sets.

### 5.6 Walk-forward validation

A single train/test split gives you one number from one market regime. Walk-forward
repeats it on consecutive periods, always training on the past.

In [ ]:
from seq_dataset import (load_prices, build_stock_datasets, baseline_scores,
                         skill, walk_forward)
from seq_train import train

prices = load_prices("AAPL", start="2015-01-01")
rows = []
for k, prices_k, fracs in walk_forward(prices, n_folds=4, val_len=250, test_len=250):
    tr, va, te, scaler = build_stock_datasets(
        prices_k, target="volatility", window=30, fracs=fracs)
    base = baseline_scores(te, scaler, tr)
    cfg = dict(run_name=f"wf_vol_lstm_fold{k}", task="regression", cell="lstm",
               input_size=2, out_features=1, hidden_size=32, lr=1e-3,
               weight_decay=1e-4, clip=1.0, epochs=40, patience=8)
    res = train(cfg, tr, va, te, extras=scaler)
    rows.append((k, skill(res["test_loss"], base["recent_vol_mse"]),
                 skill(res["test_loss"], base["train_mean_mse"])))
    print(rows[-1])

**Task.** Report skill per fold against **both** baselines, with mean and standard
deviation. Does skill stay positive in every fold, or depend on the period? Why must each
fold's scaler be **refitted** rather than reused from fold 0?

---
## 6️⃣ Inference: Loading `best.pt`

Download the most recent prices, build the latest window exactly as in training, predict.

> 💡 The scaler lives in the checkpoint's `extras`. Forgetting to apply it — or to
> **invert** the target scaling at the end — is the classic way to get a forecast that is
> "off by a constant factor".

In [ ]:
"""predict_stock.py - predict from the latest data using a saved best.pt"""

import numpy as np
import pandas as pd
import torch
import yfinance as yf

from seq_model import SeqModel

CHECKPOINT_PATH = "checkpoints/stock_AAPL_volatility_lstm/best.pt"   # hardcoded - change
TICKER = "AAPL"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
config, scaler = ckpt["config"], ckpt["extras"]
window, target = scaler["window"], scaler["target"]
print(f"Loaded epoch {ckpt['epoch']} (val_loss={ckpt['val_loss']:.4f}), target = {target}")

# Rebuild the SAME architecture the checkpoint was trained with.
model = SeqModel(
    cell=config["cell"], input_size=config["input_size"],
    hidden_size=config.get("hidden_size", 64),
    num_layers=config.get("num_layers", 1),
    bidirectional=config.get("bidirectional", False),
    dropout=config.get("dropout", 0.0),
    out_features=config["out_features"],
    pooling=config.get("pooling", "last"),
).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()                      # disables dropout - NOT optional

# ---- download recent data and preprocess EXACTLY as in training ----
raw = yf.download(TICKER, period="1y", auto_adjust=True, progress=False)
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)
close = raw["Close"].astype(float)
volume = raw["Volume"].astype(float).replace(0, np.nan)
frame = pd.DataFrame({"ret": np.log(close).diff(), "dvol": np.log(volume).diff()})
frame = frame.replace([np.inf, -np.inf], np.nan).dropna()
if len(frame) < window:
    raise ValueError(f"need at least {window} days, got {len(frame)}")

scale = np.array([scaler["ret_std"], scaler["dvol_std"]])       # the TRAIN-fitted scale
feats = np.clip(frame[["ret", "dvol"]].to_numpy() / scale, -8, 8).astype(np.float32)
x = torch.tensor(feats[-window:][None], device=device)          # (1, window, 2)
print(f"Latest data point used: {frame.index[-1].date()}")

with torch.no_grad():
    out = model(x)

if target == "volatility":
    # undo the target scaling, then undo the log
    daily_vol = float(np.exp(out.item() * scaler["t_std"] + scaler["t_mean"]))
    print(f"Predicted daily volatility over the next {scaler['h_vol']} days: "
          f"{daily_vol * 100:.2f}%   (annualised ~ {daily_vol * np.sqrt(252) * 100:.1f}%)")
elif target == "direction":
    prob = torch.softmax(out, dim=1)[0]             # softmax at display time, not in forward()
    print(f"P(up tomorrow) = {prob[1]:.3f}   P(down) = {prob[0]:.3f}")
else:
    pred_ret = out.item() * scaler["t_std"]
    print(f"Predicted next-day log-return: {pred_ret * 100:.3f}%")

---
## 🧪 Student Exercise (Evaluative) — Is Volatility Forecastable?

Full description in [`labsheet7.md`](https://github.com/Aritra/Computer_Vision/blob/main/labsheets/labsheet7.md). Summary of what to produce:

**Part 1 — three tickers** from different markets. For each: ticker, date range, trading
days, download date; the persistence-illusion R²; and the `r_t` vs `|r_t|` autocorrelation.

**Part 2 — the grid** on your primary ticker, volatility target: cell (RNN/LSTM/GRU) ×
depth (1/2) × direction (uni/bi) × 3 seeds = 36 runs. Other two tickers: best config per
cell only.

**Required baselines in every table:** train-mean, recent-vol persistence, **a Lab 6
flattened-window MLP**, and **one stronger finance baseline** (EWMA with λ=0.94, or linear
regression on lagged `|r|`).

**Also required:** walk-forward on your best config; the negative controls (return and
direction targets); and the horizon sweep.

> 💡 **On the horizon sweep (requirement 9).** Realized volatility here is the **RMS**
> of the next `h` returns, not their standard deviation. That matters at `h_vol = 1`:
> `std()` of a single return is exactly `0`, which would collapse every target to the
> constant `log(1e-8)`, whereas RMS reduces cleanly to `log|r_{t+1}|` — the standard
> one-day volatility proxy. Sanity-check your targets at each horizon before trusting the
> sweep; the cell below does that.

In [ ]:
# Sanity-check the target at every horizon BEFORE running the sweep.
# A healthy target has many distinct values and a sensible t_std. If you ever
# see "unique=1" and t_std near zero, the target has collapsed to a constant
# and every metric computed from it is meaningless.
import numpy as np
from seq_dataset import build_stock_datasets, baseline_scores

header = ("h_vol", "unique", "t_std", "train_mean", "recent_vol")
print("{:>6s} {:>8s} {:>9s} {:>11s} {:>11s}".format(*header))
for h in (1, 5, 20):
    tr_h, _, te_h, sc_h = build_stock_datasets(df, target="volatility", window=30, h_vol=h)
    y = tr_h.targets.numpy().ravel()
    b = baseline_scores(te_h, sc_h, tr_h)
    print("{:6d} {:8d} {:9.4f} {:11.4f} {:11.4f}".format(
        h, len(np.unique(y)), sc_h["t_std"],
        b["train_mean_mse"], b["recent_vol_mse"]))

### Starter — the Lab 6 flattened-window MLP baseline

An LSTM that cannot beat an MLP on the same window is not earning its cost. The window is
`(30, 2)`, so flattened it is 60 features.

In [ ]:
import torch
from torch import nn


class FlatMLP(nn.Module):
    """Lab 6's MLP, applied to a flattened (T, F) window -> T*F features."""

    def __init__(self, seq_len, input_size, hidden=64, out_features=1, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(seq_len * input_size, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, out_features),
        )

    def forward(self, x):
        return self.net(x)


# TODO: train FlatMLP on the volatility windows and report its skill alongside
# the recurrent models. seq_train.train() builds a SeqModel internally, so either
# write a short loop here or refactor train() to accept a model factory.

### Starter — EWMA volatility baseline

`σ²_t = λ σ²_{t−1} + (1 − λ) r²_t` with `λ = 0.94` (the RiskMetrics convention), rolled
across the window, then converted to the same scaled target so it is comparable.

In [ ]:
def ewma_baseline_mse(test_ds, scaler, lam=0.94):
    """EWMA(lam) volatility forecast, scored on the SAME scaled target as the models."""
    X, y = test_ds.features, test_ds.targets
    raw = X[:, :, 0] * scaler["ret_std"]            # undo feature scaling -> raw returns
    var = raw[:, 0] ** 2
    for t in range(1, raw.shape[1]):
        var = lam * var + (1 - lam) * raw[:, t] ** 2
    pred = (torch.log(var.sqrt() + 1e-8) - scaler["t_mean"]) / scaler["t_std"]
    return ((pred[:, None] - y) ** 2).mean().item()


tr_v, va_v, te_v, sc_v = build_stock_datasets(df, target="volatility", window=30)
base_v = baseline_scores(te_v, sc_v, tr_v)
print("train_mean  MSE:", round(base_v["train_mean_mse"], 4))
print("recent_vol  MSE:", round(base_v["recent_vol_mse"], 4))
print("EWMA(0.94)  MSE:", round(ewma_baseline_mse(te_v, sc_v), 4))

# NOTE: EWMA seeded from a single squared return inside a 30-day window is a weak
# start. A fairer version warms the EWMA up on a longer history before the window.
# Say which you used.

### Submission

Submit the `best.pt` for your best volatility configuration, plus a `predict_stock.py` run
showing the printed prediction **and the date of the latest data point used**.

> 📌 No reference solution is included.

### Some things to think about

- Look at the **gradient-norm curves** of your RNN runs, not just the metric.
- Notice how much of this lab was **evaluation discipline** — splits, scalers, baselines,
  seeds — rather than architecture. In noisy domains the discipline is most of the work.
- A sequence model processes a series **one step at a time**: step `t` cannot start until
  `t−1` finishes. That makes it slow compared with architectures that look at all
  positions in parallel. Keep this in mind as the course moves on.